In [ ]:
# Install all required open-source libraries
!pip install -q diffusers transformers accelerate moviepy gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 14.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.27.2 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


In [ ]:
import os

# Ensure project directories exist
os.makedirs("scripts", exist_ok=True)
os.makedirs("audio", exist_ok=True)
os.makedirs("images", exist_ok=True)

print("Project directories initialized:")
print("- /scripts (for script files)")
print("- /audio (for voiceovers)")
print("- /images (for Stable Diffusion output images)")

Project directories initialized:
- /scripts (for script files)
- /audio (for voiceovers)
- /images (for Stable Diffusion output images)


In [ ]:
# Step 1: Mock Script Generation
topic = input("Enter topic for your educational video: ")

script = f"""[Hook]
Have you ever wondered how {topic} actually works? It's simpler than you think!

[Explanation]
At its core, {topic} is all about solving real-world challenges. It allows us to process information, make smart decisions, and automate complex tasks. Whether you're a beginner or an expert, understanding {topic} opens up a world of possibilities in today's tech-driven landscape.

[Call to Action]
If you want to master {topic}, hit that follow button and stay tuned for our next video!"""

print("\n--- Generated Script ---\n")
print(script)

with open("scripts/script.txt", "w", encoding="utf-8") as f:
    f.write(script)
print("\nSaved script to scripts/script.txt")

Enter topic for your educational video: Nueral Network

--- Generated Script ---

[Hook]
Have you ever wondered how Nueral Network actually works? It's simpler than you think!

[Explanation]
At its core, Nueral Network is all about solving real-world challenges. It allows us to process information, make smart decisions, and automate complex tasks. Whether you're a beginner or an expert, understanding Nueral Network opens up a world of possibilities in today's tech-driven landscape.

[Call to Action]
If you want to master Nueral Network, hit that follow button and stay tuned for our next video!

Saved script to scripts/script.txt


In [ ]:
# Step 2: Voice Generation (using gTTS - Google Text-to-Speech)
from gtts import gTTS

print("Generating voiceover using free Google TTS...")

with open("scripts/script.txt", "r", encoding="utf-8") as f:
    script_text = f.read()

# Clean section headers [Hook] etc. so the TTS voice doesn't read them out loud
clean_script = script_text.replace("[Hook]", "").replace("[Explanation]", "").replace("[Call to Action]", "")

tts = gTTS(text=clean_script, lang='en')
tts.save("audio/voice.mp3")

print("Voice generated successfully and saved to audio/voice.mp3")

Generating voiceover using free Google TTS...
Voice generated successfully and saved to audio/voice.mp3


In [ ]:
# Step 3: Offline Visual Prompts Generation
import json

print(f"Generating Stable Diffusion prompts for: {topic}...")

# Pre-formatted cinematic visual templates matching the user's topic
scenes = [
    {
        "scene": 1,
        "prompt": f"A futuristic, modern classroom where a bright holographic display explains the concept of {topic}, cinematic lighting, highly detailed"
    },
    {
        "scene": 2,
        "prompt": f"A close-up representation of {topic} using glowing neon lines, digital data networks, and floating binary code, cybernetic style"
    },
    {
        "scene": 3,
        "prompt": f"A person standing at the peak of a high mountain looking out at a technologically advanced city, representing the future of {topic}, conceptual art"
    }
]

print("\n--- Visual Prompts ---\n")
print(json.dumps(scenes, indent=2))

Generating Stable Diffusion prompts for: Nueral Network...

--- Visual Prompts ---

[
  {
    "scene": 1,
    "prompt": "A futuristic, modern classroom where a bright holographic display explains the concept of Nueral Network, cinematic lighting, highly detailed"
  },
  {
    "scene": 2,
    "prompt": "A close-up representation of Nueral Network using glowing neon lines, digital data networks, and floating binary code, cybernetic style"
  },
  {
    "scene": 3,
    "prompt": "A person standing at the peak of a high mountain looking out at a technologically advanced city, representing the future of Nueral Network, conceptual art"
  }
]


### Important: Enable GPU Runtime

It appears that your Colab environment is not currently configured to use a GPU, which is required for running the Stable Diffusion model. Please follow these steps to enable a GPU runtime:

1.  Go to the **Runtime** menu at the top of your Colab notebook.
2.  Select **Change runtime type**.
3.  In the dialog box that appears, set **Hardware accelerator** to `GPU`.
4.  Click **Save**.

After changing the runtime, you will need to re-run all the cells in the notebook, starting from the beginning, to ensure all libraries are installed and the models are loaded correctly on the GPU.

In [ ]:
# Step 4: Stable Diffusion Image Generation (on GPU)
import torch
from diffusers import StableDiffusionPipeline

print("Loading Stable Diffusion Model on GPU...")
# Load runwayml's stable-diffusion-v1-5 model
pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16)
pipe = pipe.to("cuda")

# Disable safety checker to prevent false positive black images on artistic outputs
pipe.safety_checker = None

# Generate the images
for i, scene in enumerate(scenes, 1):
    prompt_text = scene['prompt']
    full_prompt = f"{prompt_text}, highly detailed, digital art, cinematic lighting, 4k"
    print(f"\nGenerating Image {i}/3: {full_prompt[:80]}...")

    image = pipe(full_prompt, num_inference_steps=30).images[0]
    image_path = f"images/scene_{i}.png"
    image.save(image_path)
    print(f"Saved scene to {image_path}")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading Stable Diffusion Model on GPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

...

Saved scene to images/scene_1.png

Generating Image 2/3: A close-up representation of Nueral Network using glowing neon lines, digital da...


  0%|          | 0/30 [00:00<?, ?it/s]

Saved scene to images/scene_2.png

Generating Image 3/3: A person standing at the peak of a high mountain looking out at a technologicall...


  0%|          | 0/30 [00:00<?, ?it/s]

Saved scene to images/scene_3.png


In [ ]:
# Step 5: Final Video Compilation using MoviePy
from moviepy.editor import ImageClip, AudioFileClip, concatenate_videoclips

print("Compiling final video...")
audio = AudioFileClip("audio/voice.mp3")
total_duration = audio.duration
clip_duration = total_duration / 3.0

clip1 = ImageClip("images/scene_1.png").set_duration(clip_duration)
clip2 = ImageClip("images/scene_2.png").set_duration(clip_duration)
clip3 = ImageClip("images/scene_3.png").set_duration(clip_duration)

video = concatenate_videoclips([clip1, clip2, clip3], method="compose")
video = video.set_audio(audio)

output_path = "final_video.mp4"
video.write_videofile(output_path, fps=24, codec="libx264", audio_codec="aac")
print(f"\nSuccess! Final video created at {output_path}!")

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



Compiling final video...
Moviepy - Building video final_video.mp4.
MoviePy - Writing audio in final_videoTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video final_video.mp4



Moviepy - Done !
Moviepy - video ready final_video.mp4

Success! Final video created at final_video.mp4!
